In [1]:
!pip install transformers accelerate --quiet

In [2]:
!pip install bitsandbytes --quiet

In [ ]:
# downlaod the Qwen3-4B model 

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen3-4B" 

print('Looooooood ')
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    load_in_8bit=True 
)

print('done')
print("-" * 50)

Looooooood 


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

done
--------------------------------------------------


In [4]:
model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear8bitLt(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear8bitLt(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear8bitLt(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear8bitLt(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear8bitLt(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear8bitLt(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear8bitLt(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,)

In [15]:
# test of model work

In [5]:
prompt = 'write function rever list in pyton'
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.7)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

write function rever list in pyton

def reverse_list(lst):
    # your code here
    return lst[::-1]

def reverse_list(lst):
    return lst[::-1]

Yes, that's a simple and efficient way to reverse a list in Python. The slicing operation lst[::-1] creates a new list that is the reverse of the original. However, if you want to reverse the list in place (i.e., modify the original list), you can use the reverse() method:

def reverse_list(lst):
    lst.reverse()
    return lst

But note that the reverse() method modifies the list in place and returns None. So if you're using the first approach with slicing, you'll get a new list, while the second approach will modify the original list.

But


In [20]:
PROMPT_TEMPLATE = """
You are an expert video-text analysis AI specialized in hierarchical human activity recognition. You will be given a synchronized multi-modal chunk data containing visual features, hand motions, pre-calculated action probabilities, and transcribed text. 

Your job is to analyze the input context and generate a dual-level activity label: Low-Level Actions and a High-Level Task.

---
### 1. STRICT HEURISTIC WEIGHTING (CRITICAL)
When resolving conflicts between different data sources, you MUST apply the following strict hierarchy of trust:
1. HIGHEST TRUST (Weight: 80%): 'participant_text' and 'objects'. What the user explicitly says they are doing, combined with the exact object detected in the GoPro POV, is the ABSOLUTE TRUTH.
2. MEDIUM TRUST (Weight: 15%): 'hand_motion'. Use this to determine the exact physical interaction (e.g., REACH vs GRASP).
3. LOWEST TRUST (Weight: 5%): 'action_probs'. Treat 'action_probs' merely as a weak hint. If 'action_probs' contradicts the 'participant_text' or the 'active_objects', YOU MUST COMPLETELY IGNORE 'action_probs'. For example, if action_probs says TYPING but active_objects contains PEN and participant_text says "writing", the truth is WRITING.

---
### 2. ALLOWED LABELS (TAXONOMY)

You MUST select your labels ONLY from the approved lists below. Do not invent new terms.

#### A) LOW-LEVEL ACTIONS (Atomic Actions):
- Reach: REACH_OBJECT, MOVE_HAND_TO_OBJECT, HOVER_OVER_OBJECT, ALIGN_HAND, WITHDRAW_HAND, REPOSITION_HAND
- Grasp/Release: GRASP_OBJECT, TOUCH_OBJECT, HOLD_OBJECT, RELEASE_OBJECT, SWITCH_HAND, TWO_HAND_HOLD
- Pick/Place: PICK_UP_OBJECT, PUT_DOWN_OBJECT, PLACE_ON_SURFACE, PLACE_IN_CONTAINER, REMOVE_FROM_CONTAINER, STACK_OBJECT, UNSTACK_OBJECT
- Open/Close: OPEN_LID, CLOSE_LID, OPEN_DOOR, CLOSE_DOOR, OPEN_DRAWER, CLOSE_DRAWER, UNLOCK_OBJECT, LOCK_OBJECT
- Press/Control: PRESS_BUTTON, TURN_KNOB, FLIP_SWITCH, PULL_LEVER, TAP_SCREEN, SWIPE_SCREEN, HOLD_BUTTON, RELEASE_BUTTON
- Pour/Transfer: LIFT_CONTAINER, TILT_CONTAINER, POUR_LIQUID, POUR_SOLID, STOP_POUR, TRANSFER_OBJECT, EMPTY_CONTAINER, FILL_CONTAINER
- Tool Usage: PICK_TOOL, POSITION_TOOL, APPLY_TOOL, SCRAPE, CUT, STIR, MIX, SPREAD, SCOOP, WIPE, BRUSH, SHAKE, STAB, POKE
- Motion Primitives: MOVE_LEFT, MOVE_RIGHT, MOVE_FORWARD, MOVE_BACKWARD, MOVE_UP, MOVE_DOWN, ROTATE_OBJECT, SLIDE_OBJECT, PUSH_OBJECT, PULL_OBJECT
- Inspection: LOOK_AT_OBJECT, CHECK_STATE, READ_LABEL, VERIFY_POSITION, INSPECT_CONTENT, WAIT_FOR_RESPONSE
- Body-Level: STEP_FORWARD, STEP_BACKWARD, TURN_BODY, SIT_DOWN, STAND_UP, BEND, EXTEND_ARM

#### B) OBJECT-SPECIFIC LOW-LEVEL ACTIONS:
GRASP_CUP, PUT_CUP, POUR_WATER, OPEN_FRIDGE, CLOSE_FRIDGE, PICK_KNIFE, CUT_BREAD, SPREAD_JELLY, PRESS_COFFEE_BUTTON

#### C) HIGH-LEVEL TASKS (Overall Activities):
[
  "typing in keyboard",
  "writing on paper",
  "cutting paper with scissors",
  "measuring the monitor",
  "write measured screen on paper",
  "squat",
  "shred paper with shredder"
]


---
### 3. MANDATORY OUTPUT FORMAT (ZERO FILLER)
Return ONLY a valid JSON array matching the schema below. 
- DO NOT wrap the response in markdown blocks (e.g., NO ```json).
- DO NOT include any introductory text, internal reasoning, thought process, or explanations.
- If data fields like 'participant_text' are empty, rely strictly on 'objects' and 'hand_motion' to select the best-fitting taxonomy labels.
- Output MUST start with '[' and end with ']'.

[
  {
    "chunk_id": int,
    "low_level_actions": [
      {
        "verb": "EXACT_VERB_FROM_LIST",
        "object": "target_object_or_null",
        "confidence": float
      }
    ],
    "high_level_task": "EXACT_STRING_FROM_HIGH_LEVEL_LIST"
  }
]
"""

In [22]:
with open("./fd_silent_fix.json", "r", encoding="utf-8") as file:
    chunks_data = json.load(file)


# just get 2 first chunck 
sample_chunks = chunks_data[20:22] 

chunks_text = json.dumps(sample_chunks, indent=2)

final_prompt = PROMPT_TEMPLATE + chunks_text

In [17]:
# prepare prompt and input 

# i just make llm output for just to random chunk as example
# if every this is ok tell me to run model for more chuncks

In [23]:

inputs = tokenizer(final_prompt, return_tensors="pt").to("cuda")

print('starting')


with torch.no_grad():
    outputs = model.generate(
        **inputs, 
        max_new_tokens=1000, 
        
        
        do_sample=True,
        temperature=0.1,  
        
        #Prevent from getting stuck in loops
        repetition_penalty=1.15,
        
        top_p=0.9,
        
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print(response)

starting
 

---

Now, let's begin analyzing each chunk:

For each chunk, perform these steps:
1. Analyze the 'participant_text' field first. This is the most trusted source.
   - If it's non-empty, use that as the primary indicator of what the person is doing.
   - If it's empty, look at the 'objects' list for clues about what's being manipulated.
2. Then, check the 'hand_motion' field for more specific details about how the action is performed.
3. Finally, consider the 'action_probs' only if needed, but always subordinate to the other two.

Let me now go through each chunk one by one.

Chunk ID 20:
- Time: 60-63
- Objects: LAPTOP, MONITOR, KEYBOARD, PEN
- Participant Text: "writing"
- Hand Motion: "move_hand_to_object"

Based on the highest-trust source, the participant is engaged in "writing". The objects present include a PEN, which is commonly used for writing. The hand motion indicates movement towards an object, suggesting the act of reaching for the pen before starting to write.

In [31]:
import json
import re
import torch
from transformers import StoppingCriteria, StoppingCriteriaList

# Forces the model to shut down

class JsonArrayStoppingCriteria(StoppingCriteria):
    def __init__(self, tokenizer, prompt_length):
        self.tokenizer = tokenizer
        self.prompt_length = prompt_length

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        generated_text = self.tokenizer.decode(input_ids[0][self.prompt_length:], skip_special_tokens=True)
        if "]" in generated_text and (generated_text.strip().endswith("]") or "\n" in generated_text[generated_text.rfind("]"):]) :
            try:
                start_idx = generated_text.find("[")
                end_idx = generated_text.rfind("]")
                if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
                    json.loads(generated_text[start_idx:end_idx+1])
                    return True 
            except json.JSONDecodeError:
                pass
        return False


def extract_clean_json(raw_text: str) -> list:

    text = raw_text.strip()
    
    matches = re.findall(r'\[\s*\{.*\}\s*\]', text, re.DOTALL)
    if matches:
        try:
            return json.loads(matches[0])
        except json.JSONDecodeError:
            pass

    start_idx = text.find("[")
    end_idx = text.find("]")
    
    while start_idx != -1:
        if end_idx != -1 and end_idx > start_idx:
            try:
                return json.loads(text[start_idx : end_idx + 1])
            except json.JSONDecodeError:
                end_idx = text.find("]", end_idx + 1)
                if end_idx == -1:
                    break
        start_idx = text.find("[", start_idx + 1)
        end_idx = text.find("]", start_idx)

    raise ValueError("No valid JSON array could be found or parsed from model response.")

In [38]:


#  1 - config 
INPUT_FILE = "fd_silent_fix.json"


BATCH_SIZE = 1



START_INDEX = 20  
END_INDEX = 21    


with open(INPUT_FILE, "r", encoding="utf-8") as file:
    all_chunks = json.load(file)

if START_INDEX is not None or END_INDEX is not None:
    start = START_INDEX if START_INDEX is not None else 0
    end = END_INDEX if END_INDEX is not None else len(all_chunks)
    active_chunks = all_chunks[start:end]
    print(f" Processing items {start} to {end} (Total: {len(active_chunks)} items)")
else:
    active_chunks = all_chunks
    print(f" ALL items (Total: {len(active_chunks)} items)")

total_items = len(active_chunks)
total_batches = (total_items + BATCH_SIZE - 1) // BATCH_SIZE

print(f" processing in groups of {BATCH_SIZE}...")
print("=" * 60)


# 2-  Main Loop 


all_results = []

for idx, i in enumerate(range(0, total_items, BATCH_SIZE), start=1):
    batch_chunks = active_chunks[i : i + BATCH_SIZE]
    print(f"\n[Batch {idx} of {total_batches}] Processing...")
    
    
    chunks_text = json.dumps(batch_chunks, indent=2)
    final_prompt = PROMPT_TEMPLATE + chunks_text
    
    inputs = tokenizer(final_prompt, return_tensors="pt").to("cuda")
    prompt_length = inputs.input_ids.shape[1]
    
    
    stopping_criteria = StoppingCriteriaList([JsonArrayStoppingCriteria(tokenizer, prompt_length)])
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=1000, 
            do_sample=True,          
            temperature=0.1,         
            repetition_penalty=1.2, 
            top_p=0.9,
            stopping_criteria=stopping_criteria, 
            pad_token_id=tokenizer.eos_token_id
        )
    
    batch_response_text = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)
    
    print(f"--- Model Output for Batch {idx} ---")
    print(batch_response_text.strip())
    print("-" * 40)
    
    

    try:
        batch_json_data = extract_clean_json(batch_response_text)
        
        # Enforce flat array structure
        if isinstance(batch_json_data, list):
            for item in batch_json_data:
                if isinstance(item, dict):
                    all_results.append(item)
                else:
                    print(f"⚠️ Warning: Found non-dictionary item inside batch array.")
        elif isinstance(batch_json_data, dict):
            all_results.append(batch_json_data)
            
        print(f"✨ Successfully parsed Batch {idx}!")
            
    except (json.JSONDecodeError, ValueError) as e:
        print(f"⚠️ JSON Parse Error in Batch {idx}. Saving raw text instead.")
        all_results.append({
            "error": f"Failed to parse JSON: {str(e)}", 
            "raw_output": batch_response_text
        })

print("\n" + "=" * 60)
print("loop complete!")

 Processing items 20 to 21 (Total: 1 items)
 processing in groups of 1...

[Batch 1 of 1] Processing...
--- Model Output for Batch 1 ---
---

Now, let's begin processing the provided data for each chunk:

For each chunk, perform these steps:

1. **Determine the high-level task** using the `participant_text` field first. Check against the list of allowed high-level tasks. If it matches exactly, use that. Otherwise, look at the objects involved and infer based on common scenarios. The high-level task must be one of the listed options.

2. **Identify low-level actions**:  
   - Start by checking the `participant_text`. This has highest priority. Look for verbs like "writing" which maps directly to "writing on paper".  
   - Then check the `objects`, especially what’s being held or moved.  
   - Finally, consider `hand_motion` for precise motion details.  
   - Ensure all selected verbs match the allowed low-level action vocabulary.  
   - Assign confidence scores based on how well the evi

In [39]:
OUTPUT_FILE = "output_predictions.json"


with open(OUTPUT_FILE, "w", encoding="utf-8") as out_file:
    json.dump(all_results, out_file, indent=2, ensure_ascii=False)

print(f" data successfully stored : '{OUTPUT_FILE}'")

 data successfully stored : 'output_predictions.json'
